In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
    roc_auc_score,
)
import matplotlib.pyplot as plt



### downlaod and prepare the dataset

In [ ]:
import yfinance as yf

def download_and_prepare_dataset(
    tickers: list[str],
    start_date: str = "2021-01-01",
    end_date: str = "2025-01-01",
    output_dir: str = "data" ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Downloads historical adjusted closing prices from Yahoo Finance,
    computes log returns, handles missing values, and saves CSVs locally.
    """
    os.makedirs(output_dir, exist_ok=True)
    prices_path = os.path.join(output_dir, "sp500_prices.csv")
    returns_path = os.path.join(output_dir, "sp500_log_returns.csv")

    print(f"Downloading data for {len(tickers)} tickers from {start_date} to {end_date}...")
    
    # 1. Download Adjusted Close prices
    raw_data = yf.download(tickers, start=start_date, end=end_date, progress=True)
    
    # Handle yfinance multi-index column output format
    if isinstance(raw_data.columns, pd.MultiIndex):
        prices = raw_data['Adj Close'] if 'Adj Close' in raw_data else raw_data['Close']
    else:
        prices = raw_data

    # Clean missing data (forward fill then drop any remaining NaNs)
    prices = prices.ffill().dropna(axis=1, how='any')

    # 2. Calculate daily log returns: r_t = ln(P_t / P_{t-1})
    log_returns = np.log(prices / prices.shift(1)).dropna()

    # 3. Save datasets locally
    prices.to_csv(prices_path)
    log_returns.to_csv(returns_path)

    print(f"Data saved successfully to '{output_dir}/':")
    print(f"  - Prices shape:       {prices.shape}")
    print(f"  - Log returns shape:  {log_returns.shape}")

    return prices, log_returns